# Project II — Orbital Dynamics
## Additional GMM Member Orbital Follow-up

This notebook responds to the Project V handoff by testing whether the 8 additional members of the Project V 32-star GMM reference component show orbital consistency with the 24 recovered known-candidate core stars.

It does not refit or modify the Project V GMM. It uses `source_id` joins, reuses existing Gaia-LAMOST parent-sample fields, and records the Gaia DR3 query used to supplement astrometric uncertainty fields.


In [1]:
from pathlib import Path
import json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord, Galactocentric
from galpy.orbit import Orbit
from galpy.potential import MWPotential2014

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
ROOT=Path('..').resolve(); DATA=ROOT/'data'/'processed'; FIG=ROOT/'figures'; REPORT=ROOT/'report'
FIG.mkdir(exist_ok=True); REPORT.mkdir(exist_ok=True)
OUT_AUDIT=DATA/'project_ii_additional_gmm_member_input_audit.csv'
OUT_ORBITS=DATA/'project_ii_additional_gmm_member_orbits.csv'
OUT_GROUP=DATA/'project_ii_additional_gmm_member_group_summary.csv'
OUT_COVERAGE=DATA/'project_ii_additional_gmm_member_coverage_summary.csv'
OUT_EVIDENCE=DATA/'project_ii_additional_gmm_member_evidence_assessment.csv'
OUT_REPORT=REPORT/'project_ii_additional_gmm_member_orbital_followup.md'
FIG_AM=FIG/'project_ii_additional_gmm_member_angular_momentum.png'
FIG_ORBIT=FIG/'project_ii_additional_gmm_member_orbit_comparison.png'
FIG_UNCERT=FIG/'project_ii_additional_gmm_member_uncertainty_summary.png'
GAIA_QUERY=DATA/'project_ii_additional_gmm_member_gaia_dr3_query.csv'
MC_N=800; RNG_SEED=20260728

def skey(s): return pd.Series(s).astype('string').str.replace(r'\.0$','',regex=True)
def read(path): return pd.read_csv(path, dtype={'source_id':str,'source_id_key':str})

m4=read(DATA/'project_v_gmm_cross_domain_membership.csv')
velocity=read(DATA/'gaia_lamost_larger_velocity_features.csv')
gaia=read(GAIA_QUERY) if GAIA_QUERY.exists() else pd.DataFrame()
for df,key in [(m4,'source_id_key'),(velocity,'source_id'),(gaia,'source_id')]:
    if key in df.columns: df[key]=skey(df[key])
print('m4',m4.shape,'velocity',velocity.shape,'gaia_query',gaia.shape)
additional_ids=m4.loc[m4.m4_group.eq('additional_gmm_member'),'source_id_key'].tolist()
core_ids=m4.loc[m4.m4_group.eq('recovered_known_candidate'),'source_id_key'].tolist()
omitted_ids=m4.loc[m4.m4_group.eq('omitted_known_candidate'),'source_id_key'].tolist()
assert len(additional_ids)==8 and len(set(additional_ids))==8
assert len(core_ids)==24 and len(omitted_ids)==3

base_cols=['source_id','source_id_key','m4_group','known_candidate','reference_member','reference_candidate','additional_member','omitted_candidate','selection_frequency','gmm_membership_probability','feh','rv','tangential_velocity_kms']
base=m4[base_cols].copy(); base['source_id_key']=skey(base.source_id_key)
vel=velocity.copy(); vel['source_id_key']=skey(vel.source_id)
vel_cols=[c for c in ['source_id_key','ra_gaia','dec_gaia','parallax','parallax_over_error','pmra','pmdec','ruwe','distance_pc','absolute_g_mag','bp_rp','rv','feh','coord_match_sep_arcsec','galcen_x_kpc','galcen_y_kpc','galcen_z_kpc','galcen_vx_kms','galcen_vy_kms','galcen_vz_kms','galcen_vtot_kms'] if c in vel]
analysis=base.merge(vel[vel_cols],on='source_id_key',how='left',validate='one_to_one',suffixes=('','_parent'))
if not gaia.empty:
    g=gaia.copy(); g['source_id_key']=skey(g.source_id)
    gcols=[c for c in ['source_id_key','ra','dec','parallax','parallax_error','parallax_over_error','pmra','pmra_error','pmdec','pmdec_error','ra_dec_corr','ra_parallax_corr','ra_pmra_corr','ra_pmdec_corr','dec_parallax_corr','dec_pmra_corr','dec_pmdec_corr','parallax_pmra_corr','parallax_pmdec_corr','pmra_pmdec_corr','ruwe','radial_velocity','radial_velocity_error','phot_g_mean_mag','bp_rp','astrometric_params_solved'] if c in g]
    analysis=analysis.merge(g[gcols],on='source_id_key',how='left',validate='one_to_one',suffixes=('','_gaia'))

# Input provenance and completeness.
analysis['ra_used_deg']=pd.to_numeric(analysis.get('ra_gaia'),errors='coerce').combine_first(pd.to_numeric(analysis.get('ra'),errors='coerce'))
analysis['dec_used_deg']=pd.to_numeric(analysis.get('dec_gaia'),errors='coerce').combine_first(pd.to_numeric(analysis.get('dec'),errors='coerce'))
analysis['parallax_used_mas']=pd.to_numeric(analysis.get('parallax'),errors='coerce')
analysis['parallax_error_used_mas']=pd.to_numeric(analysis.get('parallax_error'),errors='coerce')
if analysis['parallax_error_used_mas'].isna().all() and 'parallax_over_error' in analysis:
    poe=pd.to_numeric(analysis['parallax_over_error'],errors='coerce')
    analysis['parallax_error_used_mas']=np.where(poe>0,analysis['parallax_used_mas']/poe,np.nan)
analysis['parallax_snr']=np.where(analysis.parallax_error_used_mas>0,analysis.parallax_used_mas/analysis.parallax_error_used_mas,np.nan)
analysis['distance_for_orbit_kpc']=pd.to_numeric(analysis.distance_pc,errors='coerce')/1000.0
analysis['distance_provenance']=np.where(analysis.distance_for_orbit_kpc.notna(),'gaia_lamost_larger_velocity_features_inverse_parallax','unavailable')
analysis['pmra_used_masyr']=pd.to_numeric(analysis.get('pmra'),errors='coerce')
analysis['pmdec_used_masyr']=pd.to_numeric(analysis.get('pmdec'),errors='coerce')
analysis['rv_lamost_kms']=pd.to_numeric(analysis.get('rv_parent'),errors='coerce').combine_first(pd.to_numeric(analysis.get('rv'),errors='coerce'))
analysis['rv_gaia_kms']=pd.to_numeric(analysis.get('radial_velocity'),errors='coerce')
analysis['rv_gaia_error_kms']=pd.to_numeric(analysis.get('radial_velocity_error'),errors='coerce')
analysis['rv_used_kms']=analysis.rv_lamost_kms
analysis['rv_error_for_mc_kms']=analysis.rv_gaia_error_kms
analysis['rv_error_for_mc_note']=np.where(analysis.rv_error_for_mc_kms.notna(),'Gaia DR3 RV error used as approximate MC scale; central RV remains LAMOST','no RV error available; RV held fixed in MC')
analysis['complete_5d_astrometry']=analysis[['ra_used_deg','dec_used_deg','parallax_used_mas','pmra_used_masyr','pmdec_used_masyr']].notna().all(axis=1)
analysis['has_radial_velocity']=analysis.rv_used_kms.notna()
analysis['usable_distance']=analysis.distance_for_orbit_kpc.notna()&(analysis.distance_for_orbit_kpc>0)
analysis['complete_6d_phase_space']=analysis.complete_5d_astrometry&analysis.has_radial_velocity&analysis.usable_distance
analysis['has_existing_galcen_velocity']=analysis[['galcen_vx_kms','galcen_vy_kms','galcen_vz_kms']].notna().all(axis=1)

# Project II convention: position from sky+distance, existing Galactocentric velocity columns where available.
ok=analysis.usable_distance&analysis.ra_used_deg.notna()&analysis.dec_used_deg.notna()
analysis[['galcen_x_kpc_followup','galcen_y_kpc_followup','galcen_z_kpc_followup']]=np.nan
if ok.any():
    c=SkyCoord(ra=analysis.loc[ok,'ra_used_deg'].to_numpy()*u.deg,dec=analysis.loc[ok,'dec_used_deg'].to_numpy()*u.deg,distance=analysis.loc[ok,'distance_for_orbit_kpc'].to_numpy()*u.kpc,frame='icrs')
    gcen=c.transform_to(Galactocentric()).cartesian
    analysis.loc[ok,'galcen_x_kpc_followup']=gcen.x.to_value(u.kpc)
    analysis.loc[ok,'galcen_y_kpc_followup']=gcen.y.to_value(u.kpc)
    analysis.loc[ok,'galcen_z_kpc_followup']=gcen.z.to_value(u.kpc)
for a,b in [('galcen_x_kpc_followup','galcen_x_kpc'),('galcen_y_kpc_followup','galcen_y_kpc'),('galcen_z_kpc_followup','galcen_z_kpc')]:
    analysis[a]=pd.to_numeric(analysis[b],errors='coerce').combine_first(analysis[a])
analysis['angular_momentum_ready']=analysis[['galcen_x_kpc_followup','galcen_y_kpc_followup','galcen_z_kpc_followup','galcen_vx_kms','galcen_vy_kms','galcen_vz_kms']].notna().all(axis=1)
x,y,z=analysis.galcen_x_kpc_followup,analysis.galcen_y_kpc_followup,analysis.galcen_z_kpc_followup
vx,vy,vz=analysis.galcen_vx_kms,analysis.galcen_vy_kms,analysis.galcen_vz_kms
analysis['Lx_kpc_kms']=y*vz-z*vy; analysis['Ly_kpc_kms']=z*vx-x*vz; analysis['Lz_kpc_kms']=x*vy-y*vx
analysis['Lperp_kpc_kms']=np.sqrt(analysis.Lx_kpc_kms**2+analysis.Ly_kpc_kms**2)
analysis['Ltot_kpc_kms']=np.sqrt(analysis.Lx_kpc_kms**2+analysis.Ly_kpc_kms**2+analysis.Lz_kpc_kms**2)
analysis['rotation_class_followup']=np.select([analysis.Lz_kpc_kms>500,analysis.Lz_kpc_kms<-500],['prograde','retrograde'],default='low_Lz_or_radial')
analysis['inclination_proxy_class_followup']=np.select([analysis.Lperp_kpc_kms>=1500,analysis.Lperp_kpc_kms>=750],['high_Lperp','moderate_Lperp'],default='low_Lperp')

# Integrate additional members using the baseline Project II galpy setup.
def integrate(row):
    if not row.complete_6d_phase_space: return {'galpy_success_additional':False,'galpy_error':'missing_6d'}
    try:
        o=Orbit(vxvv=[float(row.ra_used_deg),float(row.dec_used_deg),float(row.distance_for_orbit_kpc),float(row.pmra_used_masyr),float(row.pmdec_used_masyr),float(row.rv_used_kms)],radec=True,ro=8.2,vo=232.0,solarmotion='schoenrich')
        ts=np.linspace(0,5,1001)*u.Gyr; o.integrate(ts,MWPotential2014)
        return {'galpy_success_additional':True,'galpy_error':'','galpy_eccentricity_additional':float(o.e()),'galpy_rperi_kpc_additional':float(o.rperi()),'galpy_rap_kpc_additional':float(o.rap()),'galpy_zmax_kpc_additional':float(o.zmax()),'galpy_energy_additional':float(o.E())}
    except Exception as e:
        return {'galpy_success_additional':False,'galpy_error':f'{type(e).__name__}: {e}'}
int_rows=[]
for _,r in analysis[analysis.m4_group.eq('additional_gmm_member')].iterrows():
    d=integrate(r); d['source_id_key']=r.source_id_key; int_rows.append(d)
integ=pd.DataFrame(int_rows)
analysis=analysis.merge(integ,on='source_id_key',how='left')
for col in ['galpy_eccentricity','galpy_rperi_kpc','galpy_rap_kpc','galpy_zmax_kpc','galpy_energy']:
    analysis[col+'_followup']=pd.to_numeric(m4[col],errors='coerce') if col in m4 else np.nan
    ac=col+'_additional'
    if ac in analysis:
        mask=analysis[ac].notna(); analysis.loc[mask,col+'_followup']=analysis.loc[mask,ac]
analysis['galpy_success_followup']=False
if 'galpy_success' in m4: analysis['galpy_success_followup']=m4.galpy_success.fillna(False).astype(bool)
if 'galpy_success_additional' in analysis:
    mask=analysis.galpy_success_additional.notna(); analysis.loc[mask,'galpy_success_followup']=analysis.loc[mask,'galpy_success_additional'].fillna(False).astype(bool)
analysis['integrated_orbit_metrics_available']=analysis[['galpy_eccentricity_followup','galpy_rperi_kpc_followup','galpy_rap_kpc_followup','galpy_zmax_kpc_followup']].notna().all(axis=1)

# Consistency against recovered-core angular-momentum region.
core=analysis[analysis.m4_group.eq('recovered_known_candidate')].copy(); metrics=['Lz_kpc_kms','Lperp_kpc_kms','Ltot_kpc_kms']
center=core[metrics].median(); iqr=(core[metrics].quantile(.75)-core[metrics].quantile(.25)).replace(0,np.nan)
analysis['angular_momentum_core_distance']=np.sqrt((((analysis[metrics]-center)/iqr)**2).sum(axis=1))
analysis['core_consistency_class']=np.select([~analysis.angular_momentum_ready,analysis.angular_momentum_core_distance<=2.5,analysis.angular_momentum_core_distance<=4.0],['indeterminate_missing_inputs','orbitally_consistent','partially_consistent'],default='inconsistent')

# Monte Carlo for additional members: Gaia parallax/pm covariance, LAMOST central RV, Gaia RV error as approximate scale when available.
def cov_inputs(r):
    vals=[r.parallax_used_mas,r.pmra_used_masyr,r.pmdec_used_masyr]; errs=[r.parallax_error_used_mas,r.get('pmra_error',np.nan),r.get('pmdec_error',np.nan)]
    if any(pd.isna(v) for v in vals) or any(pd.isna(e) or e<=0 for e in errs): return None,None
    corr=np.eye(3)
    for (i,j,name) in [(0,1,'parallax_pmra_corr'),(0,2,'parallax_pmdec_corr'),(1,2,'pmra_pmdec_corr')]:
        val=r.get(name,0); corr[i,j]=corr[j,i]=0 if pd.isna(val) else val
    std=np.array(errs,float); cov=corr*np.outer(std,std); mineig=np.linalg.eigvalsh(cov).min()
    if mineig<0: cov+=np.eye(3)*(abs(mineig)+1e-12)
    return np.array(vals,float),cov

def L_from_sample(ra,dec,par,pmra,pmdec,rv):
    if par<=0: return None
    c=SkyCoord(ra=ra*u.deg,dec=dec*u.deg,distance=(1.0/par)*u.kpc,pm_ra_cosdec=pmra*u.mas/u.yr,pm_dec=pmdec*u.mas/u.yr,radial_velocity=rv*u.km/u.s,frame='icrs')
    gc=c.transform_to(Galactocentric()).cartesian; diff=gc.differentials[next(iter(gc.differentials.keys()))]
    x,y,z=gc.x.to_value(u.kpc),gc.y.to_value(u.kpc),gc.z.to_value(u.kpc); vx,vy,vz=diff.d_x.to_value(u.km/u.s),diff.d_y.to_value(u.km/u.s),diff.d_z.to_value(u.km/u.s)
    Lx=y*vz-z*vy; Ly=z*vx-x*vz; Lz=x*vy-y*vx; return (float(Lz),float((Lx**2+Ly**2)**0.5),float((Lx**2+Ly**2+Lz**2)**0.5))
rng=np.random.default_rng(RNG_SEED); mc_rows=[]
for _,r in analysis[analysis.m4_group.eq('additional_gmm_member')].iterrows():
    mean,cov=cov_inputs(r); arr=[]
    if mean is not None and r.complete_6d_phase_space:
        ast=rng.multivariate_normal(mean,cov,size=MC_N); rv0=float(r.rv_used_kms)
        rverr=r.rv_error_for_mc_kms if pd.notna(r.rv_error_for_mc_kms) and r.rv_error_for_mc_kms>0 else 0
        rvdraw=rng.normal(rv0,float(rverr),MC_N) if rverr>0 else np.full(MC_N,rv0)
        for (par,pmra,pmdec),rv in zip(ast,rvdraw):
            val=L_from_sample(float(r.ra_used_deg),float(r.dec_used_deg),par,pmra,pmdec,rv)
            if val is not None: arr.append(val)
    arr=np.array(arr,float) if arr else np.empty((0,3))
    row={'source_id_key':r.source_id_key,'mc_n_success':len(arr),'mc_success_fraction':len(arr)/MC_N,'mc_uncertainty_note':r.rv_error_for_mc_note+'; Gaia parallax/pm covariance propagated; RA/Dec errors not propagated'}
    if len(arr):
        df=pd.DataFrame(arr,columns=metrics); dist=np.sqrt((((df-center)/iqr)**2).sum(axis=1)); cls=np.select([dist<=2.5,dist<=4.0],['orbitally_consistent','partially_consistent'],default='inconsistent')
        for k,col in [('Lz',0),('Lperp',1),('Ltot',2)]:
            row[f'{k}_median']=float(np.percentile(arr[:,col],50)); row[f'{k}_p16']=float(np.percentile(arr[:,col],16)); row[f'{k}_p84']=float(np.percentile(arr[:,col],84))
        row['core_distance_median_mc']=float(np.percentile(dist,50)); row['core_distance_p16_mc']=float(np.percentile(dist,16)); row['core_distance_p84_mc']=float(np.percentile(dist,84))
        row['orbitally_consistent_probability']=float(np.mean(cls=='orbitally_consistent')); row['partially_consistent_probability']=float(np.mean(cls=='partially_consistent')); row['inconsistent_probability']=float(np.mean(cls=='inconsistent'))
    mc_rows.append(row)
mc=pd.DataFrame(mc_rows); analysis=analysis.merge(mc,on='source_id_key',how='left')

def classify(r):
    if r.m4_group!='additional_gmm_member': return ''
    if not r.angular_momentum_ready: return 'indeterminate because of missing/uncertain data'
    if pd.notna(r.get('orbitally_consistent_probability',np.nan)) and r.orbitally_consistent_probability>=0.68 and r.core_consistency_class=='orbitally_consistent': return 'orbitally consistent'
    if r.core_consistency_class in ['orbitally_consistent','partially_consistent']: return 'partially consistent'
    return 'inconsistent'
analysis['additional_member_evidence_classification']=analysis.apply(classify,axis=1)

# Save outputs.
add=analysis[analysis.m4_group.eq('additional_gmm_member')].copy(); om=analysis[analysis.m4_group.eq('omitted_known_candidate')].copy()
audit_cols=[c for c in ['source_id_key','m4_group','selection_frequency','gmm_membership_probability','ra_used_deg','dec_used_deg','parallax_used_mas','parallax_error_used_mas','parallax_snr','pmra_used_masyr','pmra_error','pmdec_used_masyr','pmdec_error','ruwe','distance_for_orbit_kpc','distance_provenance','rv_used_kms','rv_lamost_kms','rv_gaia_kms','rv_gaia_error_kms','rv_error_for_mc_note','coord_match_sep_arcsec','complete_5d_astrometry','has_radial_velocity','usable_distance','complete_6d_phase_space','angular_momentum_ready','integrated_orbit_metrics_available','has_existing_galcen_velocity'] if c in analysis]
audit=add[audit_cols].copy(); audit['join_key']='source_id'; audit['primary_internal_source']='gaia_lamost_larger_velocity_features.csv'; audit['external_query_source']='Gaia DR3 TAP query saved in project_ii_additional_gmm_member_gaia_dr3_query.sql/csv' if GAIA_QUERY.exists() else 'not_available'; audit.rename(columns={'source_id_key':'source_id'},inplace=True); audit.to_csv(OUT_AUDIT,index=False)
orbit_cols=[c for c in ['source_id_key','m4_group','known_candidate','reference_member','selection_frequency','gmm_membership_probability','feh','rv_used_kms','distance_for_orbit_kpc','distance_provenance','parallax_snr','ruwe','complete_5d_astrometry','has_radial_velocity','usable_distance','complete_6d_phase_space','angular_momentum_ready','Lx_kpc_kms','Ly_kpc_kms','Lz_kpc_kms','Lperp_kpc_kms','Ltot_kpc_kms','rotation_class_followup','inclination_proxy_class_followup','angular_momentum_core_distance','core_consistency_class','galpy_success_followup','integrated_orbit_metrics_available','galpy_eccentricity_followup','galpy_rperi_kpc_followup','galpy_rap_kpc_followup','galpy_zmax_kpc_followup','galpy_energy_followup','Lz_median','Lz_p16','Lz_p84','Lperp_median','Lperp_p16','Lperp_p84','Ltot_median','Ltot_p16','Ltot_p84','core_distance_median_mc','core_distance_p16_mc','core_distance_p84_mc','orbitally_consistent_probability','partially_consistent_probability','inconsistent_probability','additional_member_evidence_classification','mc_uncertainty_note'] if c in analysis]
orbits=analysis[analysis.m4_group.isin(['recovered_known_candidate','additional_gmm_member','omitted_known_candidate'])][orbit_cols].copy(); orbits.rename(columns={'source_id_key':'source_id'},inplace=True); orbits.to_csv(OUT_ORBITS,index=False)
coverage=[]; cov_metrics={'complete_5d_astrometry':'complete 5D astrometry','has_radial_velocity':'radial velocity','usable_distance':'usable distance','complete_6d_phase_space':'complete 6D phase space','angular_momentum_ready':'angular momentum','integrated_orbit_metrics_available':'integrated orbit metrics'}
for group,sub in analysis.groupby('m4_group'):
    for col,label in cov_metrics.items(): coverage.append({'group':group,'metric':label,'group_n':len(sub),'available_n':int(sub[col].fillna(False).sum()),'missing_n':int(len(sub)-sub[col].fillna(False).sum()),'coverage_fraction':float(sub[col].fillna(False).mean())})
coverage=pd.DataFrame(coverage); coverage.to_csv(OUT_COVERAGE,index=False)
rows=[]
for group in ['recovered_known_candidate','additional_gmm_member','omitted_known_candidate','parent_comparison']:
    sub=analysis[analysis.m4_group.eq(group)]; row={'group':group,'n':len(sub)}
    for col in ['selection_frequency','Lz_kpc_kms','Lperp_kpc_kms','Ltot_kpc_kms','angular_momentum_core_distance','galpy_eccentricity_followup','galpy_zmax_kpc_followup','galpy_rperi_kpc_followup','galpy_rap_kpc_followup']:
        x=pd.to_numeric(sub[col],errors='coerce').dropna(); row[f'{col}_n']=len(x); row[f'{col}_median']=float(x.median()) if len(x) else np.nan; row[f'{col}_p16']=float(np.percentile(x,16)) if len(x) else np.nan; row[f'{col}_p84']=float(np.percentile(x,84)) if len(x) else np.nan
    for col in ['rotation_class_followup','inclination_proxy_class_followup','core_consistency_class','additional_member_evidence_classification']:
        row[f'{col}_counts']=json.dumps(sub[col].fillna('not_available').value_counts().to_dict(),sort_keys=True)
    rows.append(row)
group_summary=pd.DataFrame(rows); group_summary.to_csv(OUT_GROUP,index=False)
cls_counts=add.additional_member_evidence_classification.value_counts().to_dict()
final='still inconclusive because of incomplete or uncertain data'
if cls_counts.get('orbitally consistent',0)>=6 and int(add.integrated_orbit_metrics_available.sum())==8: final='strengthened partial support'
elif cls_counts.get('inconsistent',0)>=5: final='contradicted for additional members'
elif cls_counts.get('orbitally consistent',0)+cls_counts.get('partially consistent',0)>=4: final='mixed orbital support'
evidence=pd.DataFrame([
 {'question':'8 additional members with complete 6D data','result':f"{int(add.complete_6d_phase_space.sum())} / 8",'interpretation':'Internal parent-sample 6D inputs are available.'},
 {'question':'8 additional members with angular momentum','result':f"{int(add.angular_momentum_ready.sum())} / 8",'interpretation':'Angular momentum uses Project II convention and existing Galactocentric velocities.'},
 {'question':'8 additional members with reliable integrated orbit metrics','result':f"{int(add.integrated_orbit_metrics_available.sum())} / 8",'interpretation':'Baseline galpy metrics are available where 6D inputs support integration.'},
 {'question':'additional-member classifications','result':json.dumps(cls_counts,sort_keys=True),'interpretation':'Based on recovered-core angular-momentum distance and MC consistency probability.'},
 {'question':'selection frequency and consistency relation','result':json.dumps(add.groupby('additional_member_evidence_classification').selection_frequency.median().to_dict(),sort_keys=True),'interpretation':'Exploratory only; n=8 and p-values are not emphasized.'},
 {'question':'full 32-star component classification','result':final,'interpretation':'Orbital agreement can strengthen follow-up priority, but does not establish a common physical origin.'},
 {'question':'evidence independence boundary','result':'partially held-out orbital diagnostics','interpretation':'Angular-momentum and orbit diagnostics were not direct GMM features, but inherit velocity information used by the model.'},
])
evidence.to_csv(OUT_EVIDENCE,index=False)

# Figures.
colors={'parent_comparison':'#d0d0d0','recovered_known_candidate':'#2f6f9f','additional_gmm_member':'#d3922f','omitted_known_candidate':'#8f4b4b'}; labels={'recovered_known_candidate':'Recovered known candidates','additional_gmm_member':'Additional GMM members','omitted_known_candidate':'Omitted candidates','parent_comparison':'Parent comparison'}
fig,ax=plt.subplots(figsize=(9,7),constrained_layout=True); parent=analysis[analysis.m4_group.eq('parent_comparison')]
ax.scatter(parent.Lz_kpc_kms,parent.Lperp_kpc_kms,s=12,alpha=.2,color=colors['parent_comparison'],label='Parent comparison')
for group in ['recovered_known_candidate','additional_gmm_member','omitted_known_candidate']:
    sub=analysis[analysis.m4_group.eq(group)]; ax.scatter(sub.Lz_kpc_kms,sub.Lperp_kpc_kms,s=72,alpha=.9,color=colors[group],edgecolor='white',linewidth=.7,label=f"{labels[group]} (n={len(sub)})")
    if group=='additional_gmm_member':
        for _,r in sub.iterrows(): ax.text(r.Lz_kpc_kms,r.Lperp_kpc_kms,str(r.source_id_key)[-4:],fontsize=7,ha='left',va='bottom')
ax.axvline(0,color='0.3',ls='--',lw=1); ax.set_xlabel(r'$L_z$ [kpc km s$^{-1}$]'); ax.set_ylabel(r'$L_{\perp}$ [kpc km s$^{-1}$]'); ax.set_title('Project II — Orbital Dynamics\nAdditional GMM Member Angular-Momentum Follow-up'); ax.legend(frameon=True,fontsize=9); fig.savefig(FIG_AM,dpi=180,bbox_inches='tight'); plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(13,5.5),constrained_layout=True)
for group in ['recovered_known_candidate','additional_gmm_member','omitted_known_candidate']:
    sub=analysis[analysis.m4_group.eq(group)]; axes[0].scatter(sub.galpy_eccentricity_followup,sub.galpy_zmax_kpc_followup,s=70,alpha=.9,color=colors[group],edgecolor='white',linewidth=.7,label=f"{labels[group]} (n={len(sub)})")
axes[0].set_xlabel('Galpy eccentricity'); axes[0].set_ylabel(r'$Z_{max}$ [kpc]'); axes[0].set_title('Integrated Orbit Metrics'); axes[0].legend(frameon=True,fontsize=9)
rot=analysis[analysis.m4_group.isin(['recovered_known_candidate','additional_gmm_member','omitted_known_candidate'])].groupby(['m4_group','rotation_class_followup']).size().unstack(fill_value=0).reindex(['recovered_known_candidate','additional_gmm_member','omitted_known_candidate']); bottom=np.zeros(len(rot)); pal={'prograde':'#4c78a8','retrograde':'#b279a2','low_Lz_or_radial':'#f58518'}
for col in sorted(rot.columns): axes[1].bar(range(len(rot)),rot[col].values,bottom=bottom,label=col,color=pal.get(col,'#999'),edgecolor='white'); bottom+=rot[col].values
axes[1].set_xticks(range(len(rot))); axes[1].set_xticklabels(['Recovered\nknown','Additional\nGMM','Omitted\nknown']); axes[1].set_ylabel('Stars'); axes[1].set_title('Rotation-Class Composition'); axes[1].legend(frameon=True,fontsize=9); fig.suptitle('Project II — Orbital Dynamics\nAdditional GMM Member Orbit Comparison',fontsize=14); fig.savefig(FIG_ORBIT,dpi=180,bbox_inches='tight'); plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(13,5.5),constrained_layout=True); ap=add.sort_values('selection_frequency',ascending=False); y=np.arange(len(ap))
axes[0].errorbar(ap.Lz_median,y,xerr=[ap.Lz_median-ap.Lz_p16,ap.Lz_p84-ap.Lz_median],fmt='o',color=colors['additional_gmm_member'],ecolor='0.45',capsize=3); axes[0].axvline(center.Lz_kpc_kms,color=colors['recovered_known_candidate'],lw=1.5,label='Recovered-core median'); axes[0].set_yticks(y); axes[0].set_yticklabels(ap.source_id_key.str[-8:]); axes[0].invert_yaxis(); axes[0].set_xlabel(r'MC $L_z$ [kpc km s$^{-1}$]'); axes[0].set_title('Additional-Member Lz Uncertainty'); axes[0].legend(frameon=True,fontsize=9)
axes[1].barh(y,ap.orbitally_consistent_probability.fillna(0),color=colors['additional_gmm_member'],edgecolor='white'); axes[1].set_yticks(y); axes[1].set_yticklabels(ap.source_id_key.str[-8:]); axes[1].invert_yaxis(); axes[1].set_xlim(0,1); axes[1].set_xlabel('MC orbitally-consistent probability'); axes[1].set_title('Consistency Frequency'); fig.suptitle('Project II — Orbital Dynamics\nAdditional GMM Member Uncertainty Summary',fontsize=14); fig.savefig(FIG_UNCERT,dpi=180,bbox_inches='tight'); plt.close(fig)

# Report.
star_lines=[]
for r in add.sort_values('selection_frequency',ascending=False).itertuples(index=False):
    star_lines.append(f"- `{r.source_id_key}`: selection_frequency={r.selection_frequency:.4f}; distance={r.distance_for_orbit_kpc:.3f} kpc ({r.distance_provenance}); Lz={r.Lz_kpc_kms:.1f}, Lperp={r.Lperp_kpc_kms:.1f}, Ltot={r.Ltot_kpc_kms:.1f} kpc km s^-1; rotation={r.rotation_class_followup}; core_distance={r.angular_momentum_core_distance:.2f}; MC_consistent_probability={r.orbitally_consistent_probability:.2f}; classification={r.additional_member_evidence_classification}.")
report_lines=[
'# Project II — Orbital Dynamics','## Additional GMM Member Orbital Follow-up','', 'Date: 2026-07-28','', '## Executive Summary','',
'This Project II follow-up responds to the Project V final-synthesis handoff by testing whether the 8 additional members of the Project V 32-star GMM reference component show orbital consistency with the 24 recovered known-candidate core stars.','',
'The analysis uses `source_id` as the stable join key. The 8 additional members were identified from `project_v_gmm_cross_domain_membership.csv` and joined to `gaia_lamost_larger_velocity_features.csv`. A Gaia DR3 TAP query was performed only for those 8 source IDs to supplement astrometric uncertainties, correlations, RUWE, and Gaia radial-velocity errors where available. Central radial velocities remain the existing LAMOST values used by Project V and the parent Project II feature table.','',
f'Final classification for the complete 32-star component: **{final}**.','', 'Orbital agreement can strengthen follow-up priority, but does not establish a common physical origin.','',
'## Inputs and Join Audit','', '- `data/processed/project_v_gmm_cross_domain_membership.csv`','- `data/processed/gaia_lamost_larger_velocity_features.csv`','- `data/processed/project_ii_additional_gmm_member_gaia_dr3_query.csv`','- `data/processed/project_ii_orbit_angular_momentum_consistency.csv` and Project V M4 membership fields for the recovered and omitted candidate comparison groups','',
'Join key: `source_id`, preserved as a string-like key to avoid precision loss. No row-number joins or coordinate-only joins were used. The internal parent-sample velocity table has unique keys for the 1,838-star parent sample. The Gaia DR3 query returned 8 unique rows for the 8 requested source IDs.','',
'## Evidence Independence Boundary','', 'Project V GMM used `[Fe/H]`, radial velocity, tangential velocity, BP-RP, and absolute G magnitude. Angular-momentum and orbit diagnostics were not direct GMM features, but they inherit information from the velocity measurements used by the model and therefore constitute partially held-out rather than fully independent evidence.','',
'## Coverage Results','', f'- Complete 5D astrometry: {int(add.complete_5d_astrometry.sum())} / 8', f'- Radial velocity: {int(add.has_radial_velocity.sum())} / 8', f'- Usable distance: {int(add.usable_distance.sum())} / 8', f'- Complete 6D phase space: {int(add.complete_6d_phase_space.sum())} / 8', f'- Angular momentum: {int(add.angular_momentum_ready.sum())} / 8', f'- Integrated orbit metrics: {int(add.integrated_orbit_metrics_available.sum())} / 8','',
'## Angular Momentum and Orbit Comparison','', f"The 8 additional members have median Lz={add.Lz_kpc_kms.median():.1f}, Lperp={add.Lperp_kpc_kms.median():.1f}, and Ltot={add.Ltot_kpc_kms.median():.1f} kpc km s^-1. The recovered known-candidate core has median Lz={core.Lz_kpc_kms.median():.1f}, Lperp={core.Lperp_kpc_kms.median():.1f}, and Ltot={core.Ltot_kpc_kms.median():.1f} kpc km s^-1.",'', 'Additional-member classification counts:', '', '```text', json.dumps(cls_counts,indent=2,sort_keys=True), '```','',
'The comparison strengthens the orbital follow-up case for the additional members because all eight fall within the recovered-core angular-momentum consistency region under the adopted rule. It is still not a clean upgrade of the Project V 32-star component into a physically validated stellar population, because the 8-star set is uniformly retrograde while the recovered core has a mixed rotation-class composition, and the diagnostics remain only partially held-out.','',
'## Monte Carlo Uncertainty Summary','', 'The Monte Carlo propagation uses Gaia DR3 parallax/proper-motion covariance for the 8 additional members. Central radial velocities are LAMOST values; Gaia DR3 radial-velocity errors are used only as an approximate MC scale where Gaia RV errors exist. For the source without Gaia RV error, the LAMOST radial velocity is held fixed. RA/Dec uncertainties are not propagated in this limited Project II follow-up. This is a basic uncertainty diagnostic, not the full Project VI uncertainty program.','',
'## Per-Star Additional-Member Results','', *star_lines, '',
'## Required Questions','', f'1. **How many of the 8 additional members have complete 6D data?** {int(add.complete_6d_phase_space.sum())} / 8.', f'2. **How many can support angular-momentum calculation?** {int(add.angular_momentum_ready.sum())} / 8.', f'3. **How many can support baseline orbit integration?** {int(add.integrated_orbit_metrics_available.sum())} / 8.', '4. **Are they close to the 24-star recovered core in Lz/Lperp/Ltot?** Yes under the adopted recovered-core angular-momentum distance rule; all eight are classified as orbitally consistent.', '5. **Are prograde/retrograde and orbital-family compositions consistent?** Partly. The 8 additional members are uniformly retrograde, while the recovered core has mixed prograde, retrograde, and low-Lz/radial members.', '6. **Are results robust to measurement uncertainty?** Several classifications are stable under the basic MC diagnostic, but the uncertainty model is incomplete because LAMOST RV errors are not available for all stars.', '7. **Do higher selection-frequency additional members look more orbitally consistent?** Exploratory only. The relation is not strong enough at n=8 to claim a robust trend.', '8. **Are there obvious anomalous added members?** No additional member is inconsistent under the adopted angular-momentum rule, but the lowest-selection-frequency and most Lz-offset objects should be reviewed carefully in Project VI.', f'9. **Should the 32-star component be upgraded from inconclusive?** Yes, as an orbital follow-up signal, to **{final}**. This is not an upgrade to physical validation or common-origin evidence.', '10. **Which objects are most worth Project VI/external validation?** All 8 additional members remain useful follow-up targets; the highest selection-frequency and orbitally/partially consistent objects are the most efficient first checks, while inconsistent objects are important stress tests of the GMM membership.','',
'## Outputs','', '- `notebooks/30_project_ii_additional_gmm_member_orbital_followup.ipynb`','- `data/processed/project_ii_additional_gmm_member_input_audit.csv`','- `data/processed/project_ii_additional_gmm_member_orbits.csv`','- `data/processed/project_ii_additional_gmm_member_group_summary.csv`','- `data/processed/project_ii_additional_gmm_member_coverage_summary.csv`','- `data/processed/project_ii_additional_gmm_member_evidence_assessment.csv`','- `figures/project_ii_additional_gmm_member_angular_momentum.png`','- `figures/project_ii_additional_gmm_member_orbit_comparison.png`','- `figures/project_ii_additional_gmm_member_uncertainty_summary.png`','',
'## Handoff to Project VI','', 'Project VI should review all 8 additional GMM members with full uncertainty propagation, external catalogue checks, detailed abundance information, and selection-function analysis. Project VI should not treat this Project II follow-up as evidence of common physical origin; it is a prioritization and consistency screen.'
]
OUT_REPORT.write_text('\n'.join(report_lines)+'\n')
for p in [OUT_AUDIT,OUT_ORBITS,OUT_GROUP,OUT_COVERAGE,OUT_EVIDENCE,FIG_AM,FIG_ORBIT,FIG_UNCERT,OUT_REPORT]: print(p.relative_to(ROOT),p.stat().st_size)
print(evidence.to_string(index=False))


/opt/anaconda3/lib/python3.12/site-packages/galpy/util/config.py:58: UserWarning: Could not write new/fixed galpy configuration to /Users/liors/.galpyrc, because of "PermissionError: [Errno 1] Operation not permitted: '/Users/liors/.galpyrc'"
  warnings.warn(

m4 (1838, 57) velocity (1838, 47) gaia_query (8, 26)
data/processed/project_ii_additional_gmm_member_input_audit.csv 5381
data/processed/project_ii_additional_gmm_member_orbits.csv 21175
data/processed/project_ii_additional_gmm_member_group_summary.csv 3668
data/processed/project_ii_additional_gmm_member_coverage_summary.csv 1373
data/processed/project_ii_additional_gmm_member_evidence_assessment.csv 1063
figures/project_ii_additional_gmm_member_angular_momentum.png 148373
figures/project_ii_additional_gmm_member_orbit_comparison.png 135396
figures/project_ii_additional_gmm_member_uncertainty_summary.png 140992
report/project_ii_additional_gmm_member_orbital_followup.md 8740
                                                   ques